# Defaqto Sales DB — 03 · Talk to your data

**Author:** Ketki Kothe · Solution Engineer, Snowflake · `Snowflake Solution Engineering`
**Workshop:** Snowflake × Defaqto, Tuesday 8 September 2026, Snowflake London

Notebook 01 built ten dynamic tables. This one puts a **semantic view** on top of
the three gold tables so the questions can be asked in English instead of SQL.

**Why this matters here.** Ben named two gaps: *"there's no visualisation layer.
There's no ability to export that data."* A semantic view closes a third one he
didn't name — the gap between having the data and being able to ask it anything
without writing a join.

Run notebook 01 first. This notebook reads the gold tables it creates.

## Set your alias

Same alias you used in notebook 01.

In [ ]:
%%sql -r dataframe_1
-- >>> THE ONLY LINE YOU EDIT IN THIS NOTEBOOK <<<
-- Use the same alias you used in notebook 01.
SET alias = 'kkothe';

SET my_schema = (SELECT 'DEFAQTO_DB.TRANSFORMED_' || UPPER($alias));

USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;
USE SCHEMA IDENTIFIER($my_schema);

-- All three gold tables must exist before the semantic view can be built.
SHOW DYNAMIC TABLES LIKE 'GOLD%' IN SCHEMA IDENTIFIER($my_schema);


## 2 · Build the semantic view

Three logical tables, one per gold table, with dimensions and metrics named the way
somebody would actually say them out loud — `insurer`, `car age`, `premium`.

**Two traps worth knowing, both hit while writing this:**

1. A ratio that combines two aggregates — `DIV0(SUM(a), SUM(b))` — is **rejected**
   inside a table-scoped metric. It has to be a **derived** metric: no table prefix,
   referencing two other metrics. Look at `abandonment_rate` below.
2. Dimension names must be unique across the **whole view**, not per table. Two
   tables both offering `quote_date` will not compile.

And the reason every ratio here is a ratio *of sums* rather than an average of
per-row rates: the moment daily volumes differ, an average of rates is simply wrong.

In [ ]:
%%sql -r dataframe_3
-- Built as a string, then executed. That is not decoration - the schema name has
-- to appear INSIDE the verified queries, and a verified query's SQL is a string
-- literal, so it cannot be concatenated or resolved with IDENTIFIER(). Every
-- @@SCHEMA@@ below becomes your own schema before Snowflake ever sees the DDL.
--
-- SET will not accept a bare function call - REPLACE(...) fails with
-- "assignment from non-constant source expression" - but it accepts a subquery,
-- which is why this is wrapped in (SELECT ...).
--
-- CREATE OR ALTER, not CREATE OR REPLACE. You will re-run this a dozen times
-- while tuning synonyms and comments, and OR REPLACE silently drops the
-- REFERENCES and SELECT grants the agent depends on. OR ALTER preserves them.
SET ddl = (SELECT REPLACE($$
CREATE OR ALTER SEMANTIC VIEW @@SCHEMA@@.DEFAQTO_INSIGHTS

  TABLES (
    funnel AS @@SCHEMA@@.GOLD_FUNNEL_DAILY
      PRIMARY KEY (QUOTE_DATE, AFFILIATE_ID)
      WITH SYNONYMS = ('funnel', 'conversion funnel', 'quote journey', 'drop off')
      COMMENT = 'Daily six-stage funnel by comparison site.',

    provider_day AS @@SCHEMA@@.GOLD_PROVIDER_DAILY
      PRIMARY KEY (QUOTE_DATE, PROVIDER_KEY, AFFILIATE_ID)
      WITH SYNONYMS = ('provider performance', 'insurer scorecard')
      COMMENT = 'Daily scorecard per insurer: appearances, prices, clicks, sales, premium, price position.',

    cohort AS @@SCHEMA@@.GOLD_COHORT_CONVERSION
      PRIMARY KEY (PROVIDER_KEY, VEHICLE_AGE_BAND, DRIVER_AGE_BAND, COVER_LENGTH_BAND, COVER_REASON)
      WITH SYNONYMS = ('cohort', 'customer type', 'demographics', 'segment')
      COMMENT = 'Conversion by insurer and customer type.'
  )

  DIMENSIONS (
    funnel.quote_date AS QUOTE_DATE
      WITH SYNONYMS = ('date', 'day')
      COMMENT = 'Date the quote journey started',
    funnel.comparison_site AS AFFILIATE_ID
      WITH SYNONYMS = ('affiliate', 'PCW', 'price comparison website', 'partner')
      COMMENT = 'Comparison site the quote came from',

    -- Named provider_date, not quote_date: dimension names are unique across the
    -- WHOLE view, so a second `quote_date` would fail to compile.
    provider_day.provider_date AS QUOTE_DATE
      WITH SYNONYMS = ('date', 'day')
      COMMENT = 'Date the quote journey started',
    provider_day.provider AS PROVIDER_KEY
      WITH SYNONYMS = ('insurer', 'provider', 'underwriter', 'company')
      COMMENT = 'Canonical insurer name, resolved across the quote and sales systems',
    provider_day.provider_comparison_site AS AFFILIATE_ID
      WITH SYNONYMS = ('affiliate', 'PCW', 'partner')
      COMMENT = 'Comparison site the quote came from',

    cohort.cohort_provider AS PROVIDER_KEY
      WITH SYNONYMS = ('insurer', 'provider', 'company')
      COMMENT = 'Canonical insurer name',
    cohort.vehicle_age_band AS VEHICLE_AGE_BAND
      WITH SYNONYMS = ('car age', 'vehicle age', 'age of car')
      COMMENT = 'Under 7 years, 7 years or older, or unknown'
      SAMPLE_VALUES ('under 7 years', '7 years or older', 'unknown')
      IS_ENUM,
    cohort.driver_age_band AS DRIVER_AGE_BAND
      WITH SYNONYMS = ('driver age', 'customer age', 'age group')
      COMMENT = 'Driver age band'
      SAMPLE_VALUES ('17-24', '25-34', '35-49', '50-64', '65+', 'unknown')
      IS_ENUM,
    cohort.cover_length_band AS COVER_LENGTH_BAND
      WITH SYNONYMS = ('cover length', 'duration', 'policy length')
      COMMENT = 'How long the cover was for'
      SAMPLE_VALUES ('hours', '1-3 days', '4-7 days', '8-28 days', 'unknown')
      IS_ENUM,
    cohort.cover_reason AS COVER_REASON
      WITH SYNONYMS = ('reason for cover', 'why they need cover')
      COMMENT = 'Stated reason the customer needed short-term cover',
    cohort.fully_attributed AS IS_FULLY_ATTRIBUTED
      WITH SYNONYMS = ('complete data', 'known cohort')
      COMMENT = 'TRUE when vehicle, driver and cover details are all known'
  )

  METRICS (
    funnel.quotes_started AS SUM(QUOTES_STARTED)
      WITH SYNONYMS = ('quotes', 'journeys', 'volume')
      COMMENT = 'Quote journeys started',
    funnel.reached_results AS SUM(REACHED_RESULTS)
      COMMENT = 'Journeys that reached the results page and saw prices',
    -- Stage 3. Distinct from received_a_price: a provider can respond and still
    -- decline (STATUS 3), which carries PRICE 0. 45,273 journeys - 15.2% - sit
    -- between these two metrics, and without this one that loss is invisible.
    funnel.received_a_rate AS SUM(RECEIVED_A_RATE)
      WITH SYNONYMS = ('got a response', 'provider responded', 'any rate returned')
      COMMENT = 'Journeys where at least one insurer responded at all, including declines',
    funnel.received_a_price AS SUM(RECEIVED_A_PRICE)
      WITH SYNONYMS = ('got a usable price', 'priced')
      COMMENT = 'Journeys that received at least one usable price - excludes declines priced at zero',
    funnel.clicked_out AS SUM(CLICKED_OUT)
      WITH SYNONYMS = ('clicks', 'click outs')
      COMMENT = 'Journeys where the shopper clicked through to an insurer',
    funnel.converted AS SUM(CONVERTED)
      WITH SYNONYMS = ('sales', 'conversions')
      COMMENT = 'Journeys that became a sale',
    funnel.abandoned_before_results AS SUM(ABANDONED_BEFORE_RESULTS)
      WITH SYNONYMS = ('abandonment', 'drop off before prices', 'lost early')
      COMMENT = 'Journeys abandoned BEFORE any insurer was asked - the largest single leak',

    provider_day.appearances AS SUM(QUOTES_APPEARED_IN)
      WITH SYNONYMS = ('appearances', 'times shown')
      COMMENT = 'Quotes this insurer appeared in',
    provider_day.quotes_priced AS SUM(QUOTES_PRICED)
      COMMENT = 'Quotes where this insurer returned a usable price',
    provider_day.clicks AS SUM(CLICKS)
      COMMENT = 'Click-outs to this insurer',
    provider_day.sales AS SUM(SALES)
      COMMENT = 'Sales won by this insurer',
    provider_day.gwp AS SUM(GWP)
      WITH SYNONYMS = ('premium', 'gross written premium', 'revenue')
      COMMENT = 'Gross written premium',
    provider_day.commission AS SUM(COMMISSION)
      COMMENT = 'Commission earned',
    provider_day.times_cheapest AS SUM(TIMES_CHEAPEST)
      WITH SYNONYMS = ('cheapest', 'lowest price wins')
      COMMENT = 'Quotes where this insurer was the cheapest price shown',
    provider_day.avg_price_rank AS AVG(AVG_PRICE_RANK)
      WITH SYNONYMS = ('price position', 'rank on page')
      COMMENT = 'Average position on the results page, 1 being cheapest',

    cohort.cohort_appearances AS SUM(QUOTES_APPEARED_IN)
      COMMENT = 'Quotes in this cohort the insurer appeared in',
    cohort.cohort_priced AS SUM(QUOTES_PRICED)
      COMMENT = 'Quotes in this cohort the insurer priced',
    cohort.cohort_clicks AS SUM(CLICKS)
      COMMENT = 'Click-outs from this cohort',
    cohort.cohort_sales AS SUM(SALES)
      COMMENT = 'Sales from this cohort',
    cohort.cohort_gwp AS SUM(GWP)
      COMMENT = 'Premium from this cohort',
    cohort.avg_best_price AS AVG(AVG_BEST_PRICE)
      COMMENT = 'Average cheapest price this insurer offered to this cohort',

    -- DERIVED metrics. A ratio of two aggregates cannot live in a table-scoped
    -- metric - it must be declared here with NO table prefix. Each is a ratio of
    -- sums; an average of per-row rates would be wrong once volumes differ.
    abandonment_rate AS DIV0(funnel.abandoned_before_results, funnel.quotes_started)
      WITH SYNONYMS = ('abandonment rate', 'early drop off rate')
      COMMENT = 'Share of journeys lost before any price was shown',
    overall_conversion_rate AS DIV0(funnel.converted, funnel.quotes_started)
      COMMENT = 'Sales as a share of all journeys started',
    usable_price_rate AS DIV0(funnel.received_a_price, funnel.received_a_rate)
      WITH SYNONYMS = ('price quality', 'share of responses that were usable')
      COMMENT = 'Of journeys where an insurer responded, the share that got a real price rather than a decline',
    provider_click_through_rate AS DIV0(provider_day.clicks, provider_day.quotes_priced)
      COMMENT = 'Clicks as a share of quotes this insurer priced',
    provider_win_rate AS DIV0(provider_day.sales, provider_day.clicks)
      COMMENT = 'Sales as a share of click-outs',
    provider_cheapest_share AS DIV0(provider_day.times_cheapest, provider_day.quotes_priced)
      COMMENT = 'How often this insurer was the cheapest price shown',
    cohort_click_rate AS DIV0(cohort.cohort_clicks, cohort.cohort_priced)
      WITH SYNONYMS = ('conversion by customer type', 'cohort conversion')
      COMMENT = 'Clicks as a share of priced quotes - the number to compare across cohorts'
  )

  COMMENT = 'Ask questions of the Defaqto short-term car funnel in plain English.'

  AI_SQL_GENERATION 'Round rates to one decimal place and express them as percentages by multiplying by 100. When a question is about where customers are lost, prefer the funnel table and lead with abandoned_before_results, because most loss happens before any insurer is asked for a price. Note that received_a_rate and received_a_price are different stages: a provider can respond and still decline, and declines carry a price of zero. When a question names an insurer, use provider_day, unless the question is about customer types, vehicle age or driver age, in which case use cohort. For cohort comparisons always report cohort_click_rate rather than raw click counts, because cohort sizes differ.'

  AI_QUESTION_CATEGORIZATION 'This data covers short-term car insurance only. If a question asks about breakdown, gadget, home emergency, wedding or any other product line, explain that only short-term car is modelled here. This data contains no customer names, addresses or contact details, only age bands and outward postcode, so reject any question seeking to identify an individual.'

  -- Verified queries. Two jobs: they raise accuracy on the questions we actually
  -- plan to ask live, and the ones flagged ONBOARDING_QUESTION TRUE surface in the
  -- UI as suggested questions, so a first-time user is not staring at a blank box.
  -- Every SQL below was executed against this view before being pasted in.
  -- Note the doubled quotes: '' is how a literal quote is escaped inside the SQL string.
  AI_VERIFIED_QUERIES (

    where_we_lose_customers AS (
      QUESTION 'Where do we lose customers in the quote journey?'
      VERIFIED_AT 1788480000
      ONBOARDING_QUESTION TRUE
      VERIFIED_BY '(STEWARD = DATA_STEWARD)'
      SQL 'SELECT * FROM SEMANTIC_VIEW(
             @@SCHEMA@@.DEFAQTO_INSIGHTS
             METRICS funnel.quotes_started, funnel.reached_results, funnel.received_a_rate,
                     funnel.received_a_price, funnel.clicked_out, funnel.converted,
                     funnel.abandoned_before_results, abandonment_rate,
                     usable_price_rate, overall_conversion_rate
           )'
    ),

    insurer_league_table AS (
      QUESTION 'Which insurers convert best?'
      VERIFIED_AT 1788480000
      ONBOARDING_QUESTION TRUE
      VERIFIED_BY '(STEWARD = DATA_STEWARD)'
      SQL 'SELECT * FROM SEMANTIC_VIEW(
             @@SCHEMA@@.DEFAQTO_INSIGHTS
             DIMENSIONS provider_day.provider
             METRICS provider_day.quotes_priced, provider_day.clicks, provider_day.sales,
                     provider_day.gwp, provider_day.avg_price_rank,
                     provider_click_through_rate, provider_win_rate, provider_cheapest_share
           ) ORDER BY quotes_priced DESC'
    ),

    conversion_by_vehicle_age AS (
      QUESTION 'Do insurers convert newer cars better than older cars?'
      VERIFIED_AT 1788480000
      ONBOARDING_QUESTION TRUE
      VERIFIED_BY '(STEWARD = DATA_STEWARD)'
      SQL 'SELECT * FROM SEMANTIC_VIEW(
             @@SCHEMA@@.DEFAQTO_INSIGHTS
             DIMENSIONS cohort.cohort_provider, cohort.vehicle_age_band
             METRICS cohort.cohort_priced, cohort.cohort_clicks, cohort_click_rate
             WHERE cohort.vehicle_age_band <> ''unknown''
           ) ORDER BY cohort_provider, vehicle_age_band'
    ),

    funnel_by_comparison_site AS (
      QUESTION 'Which comparison site sends us the best traffic?'
      VERIFIED_AT 1788480000
      ONBOARDING_QUESTION TRUE
      VERIFIED_BY '(STEWARD = DATA_STEWARD)'
      SQL 'SELECT * FROM SEMANTIC_VIEW(
             @@SCHEMA@@.DEFAQTO_INSIGHTS
             DIMENSIONS funnel.comparison_site
             METRICS funnel.quotes_started, funnel.converted,
                     abandonment_rate, overall_conversion_rate
           ) ORDER BY quotes_started DESC'
    ),

    abandonment_trend AS (
      QUESTION 'How has the abandonment rate changed day by day?'
      VERIFIED_AT 1788480000
      ONBOARDING_QUESTION FALSE
      VERIFIED_BY '(STEWARD = DATA_STEWARD)'
      SQL 'SELECT * FROM SEMANTIC_VIEW(
             @@SCHEMA@@.DEFAQTO_INSIGHTS
             DIMENSIONS funnel.quote_date
             METRICS funnel.quotes_started, funnel.abandoned_before_results, abandonment_rate
           ) ORDER BY quote_date'
    )
  );
$$, '@@SCHEMA@@', $my_schema));

EXECUTE IMMEDIATE $ddl;

-- Prove the schema really was substituted. If this returns FALSE the verified
-- queries point at a schema that does not exist, and the four onboarding
-- questions will fail when a user clicks them - while this cell still reports
-- success, because verified-query SQL is never parsed at create time.
SELECT GET_DDL('SEMANTIC VIEW', $my_schema || '.DEFAQTO_INSIGHTS')
       ILIKE '%' || $my_schema || '.DEFAQTO_INSIGHTS%' AS schema_substituted_ok;


## 3 · Do the same thing in Snowsight, without SQL

Everything above has a UI equivalent, and this is the route a Defaqto analyst would
actually take.

**Create or edit a semantic view**

1. Left nav → **AI & ML** → **Cortex Analyst**
2. Click **Semantic views**, then **Create** → **Create new semantic view**
3. Pick database `DEFAQTO_DB` and your `TRANSFORMED_<alias>` schema
4. Name it, then **Select tables** → tick `GOLD_FUNNEL_DAILY`,
   `GOLD_PROVIDER_DAILY`, `GOLD_COHORT_CONVERSION`
5. On each table, choose which columns are **dimensions** and which are **metrics**.
   Snowsight guesses from the data types — check the guesses, it puts numeric IDs in
   the wrong bucket.
6. Add **synonyms** on anything a person would say differently. This is the single
   highest-value thing you can do here: `insurer` for `PROVIDER_KEY`, `car age` for
   `VEHICLE_AGE_BAND`.
7. **Save**

**Try it straight away**

8. Open the semantic view and click **Ask a question**
9. Try: *"Where do we lose the most customers?"*, then
   *"Which insurer converts cars under seven years old best?"*
10. Click **View SQL** on any answer. That is the real test — read the SQL it wrote
    and decide whether you would have written the same thing.

**Add verified queries when it gets one wrong**

11. Ask the question, correct the SQL, then **Save as verified query**
12. The next person asking that question gets your version, not a fresh guess

**Use it in an agent**

13. That is section 5 below — an agent is a separate object with its own prerequisites,
    and the prerequisites are where this goes wrong.

**What to look for**

- Ask the same question two different ways and check you get the same number
- Ask something the data cannot answer — *"how many people arrived on the site?"* —
  and confirm it says so rather than inventing an answer. There is no arrivals data.
- Ask about breakdown cover. The custom instructions should tell you only short-term
  car is modelled here.

## 4 · Build the agent in Snowsight

A semantic view answers questions. An **agent** is the thing people actually talk to: it
chooses which tool to use, keeps conversation context across turns, and can draw a chart.
The semantic view becomes one of its tools.

### Read this before you click anything

Two prerequisites cause almost every "the agent can't see my data" report, and neither
produces an obvious error message.

**1. An agent uses your DEFAULT role, not the role you have selected in the UI.**
It also needs you to have a **default warehouse**, with USAGE on it granted to that
default role. If either is missing, agent calls fail *even though your current role has
every privilege*. Check yours:

```sql
DESC USER <your_user>;   -- look at DEFAULT_ROLE and DEFAULT_WAREHOUSE
```

If either is blank, an administrator has to fix it before you go further:

```sql
ALTER USER <your_user> SET DEFAULT_ROLE = <role>, DEFAULT_WAREHOUSE = COMPUTE_WH;
GRANT USAGE ON WAREHOUSE COMPUTE_WH TO ROLE <role>;
```

### Create it

1. Left nav → **AI & ML** → **Agents** → **Create agent**
2. **Agent object name**: `DEFAQTO_ANALYST` — this is the identifier you use in SQL
3. **Display name**: `Defaqto Analyst` — this is what people see and type at
4. Pick database `DEFAQTO_DB` and your `TRANSFORMED_<alias>` schema
5. **Create agent**

At this point the agent works but has **no access to any data in your account**. Ask it
*"what is a semantic view?"* and it answers from the model's general knowledge. That is
worth doing once, so the difference is obvious when you attach the tool.

### Give it the semantic view

6. Open the agent → **Edit**
7. **Description**: *Answers questions about the short-term car insurance quote funnel,
   insurer performance, and conversion by customer type.*
8. Type each sample question from section 6 below and click **Add a question** after each
9. **Tools** → find **Cortex Analyst** → **+ Add**
10. **Name**: `Funnel_Analyst`
11. Select **Semantic view** → choose `DEFAQTO_INSIGHTS`
12. **Warehouse**: `COMPUTE_WH`
13. **Query timeout**: `60` seconds
14. **Description**: *Funnel, insurer scorecard and cohort conversion for short-term car
    insurance.* — the agent reads this to decide when to reach for the tool, so write it
    for the agent, not for a human
15. **Add**
16. Still in **Tools**, enable **Data to Chart** so it can draw rather than only tabulate
17. **Orchestration** → **Planning instructions**:
    *Use Funnel_Analyst for every question about quotes, insurers, prices, clicks, sales
    or customer types. Never answer those from general knowledge.*
18. **Response instructions**:
    *Be concise. Give percentages to one decimal place and money in pounds. When you show
    a rate, also show the two counts it came from.*
19. **Save**

Step 17 matters more than it looks. Without it the agent will sometimes answer a data
question from general knowledge — confidently, and wrongly.

### Test it

20. On the agent page, use the **playground** at the bottom
21. Ask a question, then expand the response to see **which tool it called and the SQL it
    wrote**. Read the SQL. That is the whole test.

In [ ]:
%%sql -r dataframe_8
-- Calling an agent needs a Cortex database role. CORTEX_USER is granted to PUBLIC by
-- default, so you may already have it - but that default grant is sometimes revoked,
-- and then agents fail with a privilege error that does not mention Cortex at all.
-- Granting it explicitly costs nothing and removes the doubt.
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_AGENT_USER TO ROLE ACCOUNTADMIN;

-- The agent object itself needs to be creatable in your schema.
GRANT CREATE AGENT ON SCHEMA IDENTIFIER($my_schema) TO ROLE ACCOUNTADMIN;

SHOW AGENTS IN SCHEMA IDENTIFIER($my_schema);

### The same agent in SQL, so it is reproducible

The UI is the right way to explore. SQL is the right way to ship, because you can put it
in version control and recreate it in another account. Note two things that catch people:

- The key under `tool_resources` must match the tool's `name` **exactly**. Here both are
  `Funnel_Analyst`. A mismatch fails silently — the agent simply behaves as though it has
  no tool, and answers from general knowledge instead.
- `ALTER AGENT ... SET SPECIFICATION` **replaces the whole specification**. Anything you
  leave out is deleted. Always start from `DESCRIBE AGENT` output, never from memory.

In [ ]:
%%sql -r dataframe_9
CREATE OR REPLACE AGENT DEFAQTO_ANALYST
  COMMENT = 'Short-term car insurance funnel, insurer performance and cohort conversion.'
  PROFILE = '{"display_name": "Defaqto Analyst"}'
  FROM SPECIFICATION
  $$
  models:
    orchestration: auto

  instructions:
    response: "Be concise. Give percentages to one decimal place and money in pounds. When you show a rate, also show the two counts it came from."
    orchestration: "Use Funnel_Analyst for every question about quotes, insurers, prices, clicks, sales or customer types. Never answer those from general knowledge. If a question is about a product line other than short-term car insurance, say the data does not cover it."
    sample_questions:
      - question: "Where do we lose customers in the quote journey?"
      - question: "Which insurers convert best?"
      - question: "Do insurers convert newer cars better than older cars?"
      - question: "Which comparison site sends us the best traffic?"

  tools:
    - tool_spec:
        type: "cortex_analyst_text_to_sql"
        name: "Funnel_Analyst"
        description: "Funnel, insurer scorecard and cohort conversion for short-term car insurance."
    - tool_spec:
        type: "data_to_chart"
        name: "data_to_chart"
        description: "Draws a chart from data another tool returned."

  tool_resources:
    Funnel_Analyst:
      semantic_view: "DEFAQTO_INSIGHTS"
      execution_environment:
        type: "warehouse"
        warehouse: "COMPUTE_WH"
  $$;

DESCRIBE AGENT DEFAQTO_ANALYST;

## 5 · Questions to ask it

These are grouped on purpose. The first three groups are meant to work. The last two are
meant to expose a limit — and those are the ones that build trust in a room, because an
agent that admits what it cannot do is the only kind anyone should deploy.

### Start here — the headline

- *Where do we lose customers in the quote journey?*
- *What share of journeys are abandoned before any insurer is asked for a price?*
- *Of the journeys where an insurer responded, how many got a real price rather than a
  decline?*
- *Show the funnel as a chart.*

The answer to the first is the story of the whole day: about **47%** of journeys end before
any insurer is asked. Neither Defaqto nor its partners can see that today.

### Insurer performance

- *Which insurers convert best?*
- *How often is each insurer the cheapest price on the page?*
- *What is Zixty's average price position?*
- *Which insurer earns the most premium per click?*
- *Compare Zixty and CoverTime on click-through rate.*

### the cohort question — conversion by customer type

- *Do insurers convert newer cars better than older cars?*
- *Which insurer does best on cars under seven years old?*
- *Which insurer converts older cars better than newer ones?*
- *Break conversion down by driver age band for Zixty.*
- *Which cover length converts best?*

Some insurers go the other way and convert older cars better. Note that these cohort
patterns are invented in the sample data - show them as capability, never as a finding.

### Ask the same thing two ways — a consistency check

Run both and confirm the number matches. This is the fastest way to show that the semantic
view, not the model, is deciding what the metric means.

- *What percentage of quotes end in a sale?* / *What is our overall conversion rate?*
- *Which insurer gets clicked most often relative to how often it is priced?* /
  *Rank insurers by click-through rate.*

### Questions it should refuse — run these deliberately

- *How many people arrived on the site?* — **there is no arrivals data.** A journey only
  exists here once a quote is requested. If the agent invents a number, that is the single
  most important thing you will learn all day.
- *Show me breakdown cover sales.* — only short-term car is modelled. The custom
  instructions should say so.
- *Which customers live in SW1A 1AA?* — the data has outward postcode and age bands only,
  no individuals. It should decline.
- *What will conversion be next month?* — there is no forecast here. It should not guess.

### The trap worth showing last

- *What is the total number of distinct quotes across all providers?*

`DISTINCT_QUOTES` is **non-additive** — summing it across providers overstates by 2.0%,
208,384 against a true 204,311, because one quote shown to five insurers is counted five
times. This is the cleanest single argument for why the semantic view exists rather than
leaving every analyst to write their own SQL.

### How to judge any answer

Always expand the response and read the SQL. Three things to check:

1. Did it use the tool, or answer from general knowledge?
2. Is it dividing sums, or averaging per-row rates? Those differ once volumes differ.
3. Did it silently drop the `unknown` cohort rows, or say it excluded them?

## 6 · Or have Cortex Code build it

Paste this instead of running the cells:

> I have three gold dynamic tables in my schema: a daily funnel, a daily per-insurer
> scorecard, and conversion by customer cohort. Build me a semantic view over them so
> I can ask questions in English. Name dimensions and metrics the way an insurance
> analyst would say them out loud, add synonyms, and make every rate a ratio of sums
> rather than an average of rates. Add custom instructions so it refuses questions
> about product lines I don't have data for, and tell me which of my measures are
> non-additive before you write anything.

The last clause is the one that earns its keep — it makes the model tell you where
the traps are instead of quietly stepping in one.

## Done

You now have a semantic view over the gold layer, queryable in SQL through
`SEMANTIC_VIEW(...)`, in English through Cortex Analyst, and available to a Cortex
Agent in Snowflake Intelligence.

**The three things worth remembering:**

- `DISTINCT_QUOTES` cannot be summed. A semantic view is where you say that once,
  instead of hoping every analyst remembers.
- A ratio of two aggregates must be a **derived** metric, with no table prefix.
- Synonyms are not decoration. `insurer`, `car age`, `premium` are what people say;
  `PROVIDER_KEY`, `VEHICLE_AGE_BAND`, `GWP` are what the columns are called.

---

*Ketki Kothe · Solution Engineer, Snowflake · `Snowflake Solution Engineering`*
*Aggregation logic supplied by the Defaqto data lead. Sample data is synthetic — the
cohort conversion pattern was put there deliberately and is not a finding about any
real insurer.*